## 1. Config

In [9]:
QUERIES = ["network", "structure", "object"]
DATA = "data/cacm"

import statistics, sys, time, timeit
sys.path.insert(0, "src")
from ir import IR, docx_line, precision_at

## 2. Index build

In [10]:
times = []
for _ in range(5):
    t0 = time.perf_counter()
    ir = IR.build(DATA)
    times.append(time.perf_counter() - t0)
build_time = statistics.median(times)
print(f"Index build time: {build_time:.6f} s (median of 5)")

Index build time: 0.235055 s (median of 5)


## 3. Stop words

In [11]:
print(f"CACM collection; {len(ir.stop_words)} words: {', '.join(ir.stop_words)}")

CACM collection; 7 words: the, of, a, and, to, is, in


## 4. Statistics

In [12]:
print(f"Tokens before preprocessing: {ir.tokens_before:,}")
print(f"Tokens after preprocessing: {ir.tokens_after:,}")
print(f"Vocabulary size: {ir.vocabulary_size:,}")
print(f"Index vs corpus: {ir.index_bytes:,} / {ir.corpus_bytes:,} bytes ({ir.index_bytes / ir.corpus_bytes:.2%})")

Tokens before preprocessing: 374,478
Tokens after preprocessing: 159,105
Vocabulary size: 11,078
Index vs corpus: 957,888 / 2,217,926 bytes (43.19%)


## 5. Test queries

In [13]:
for q in QUERIES:
    res, rel = ir.search(q), ir.relevant(q)
    print(f"{q}: {len(res)} matches, {len(rel)} relevant")
    for d, s in res[:7]:
        print(f"  {d}  {s:.4f}  {'R' if d in rel else '-'}")
    print(f"  P@3 = {precision_at(res, rel, 3):.2f}; P@7 = {precision_at(res, rel, 7):.2f}")

network: 67 matches, 46 relevant
  CACM-0757  0.6446  R
  CACM-1685  0.4972  R
  CACM-0517  0.4834  R
  CACM-1688  0.4550  R
  CACM-1687  0.4297  R
  CACM-1690  0.4297  R
  CACM-2088  0.4297  R
  P@3 = 1.00; P@7 = 1.00
structure: 187 matches, 76 relevant
  CACM-0115  0.4735  R
  CACM-1569  0.4059  R
  CACM-0463  0.3551  R
  CACM-0652  0.3551  R
  CACM-0250  0.3157  R
  CACM-3051  0.2991  R
  CACM-1737  0.2583  R
  P@3 = 1.00; P@7 = 1.00
object: 52 matches, 7 relevant
  CACM-1733  0.2576  R
  CACM-0598  0.2228  R
  CACM-1947  0.1648  R
  CACM-2495  0.1472  R
  CACM-1165  0.1308  -
  CACM-2751  0.1288  -
  CACM-1739  0.1212  -
  P@3 = 1.00; P@7 = 0.57


## 6. Answer time

In [14]:
# total s over 1000 calls == mean ms per call
answer_ms = {q: timeit.timeit(lambda: ir.search(q), number=1000) for q in QUERIES}
for q, ms in answer_ms.items():
    print(f"{q}: {ms:.4f} ms")

network: 0.0234 ms
structure: 0.0427 ms
object: 0.0092 ms


## 7. Docx

In [15]:
print(f"Index build time: {build_time:.6f} s")
print("System answer time per query:")
for q, ms in answer_ms.items():
    print(f"{q}: {ms:.4f} ms")
print(f"Stop words: CACM collection; {len(ir.stop_words)} words: {', '.join(ir.stop_words)}.")
print("Preprocessing: HTML tags, citation block, record line and date line removed; lowercase; [a-z]+ tokens; stop words removed.")
print(f"Index size vs corpus size: {ir.index_bytes:,} / {ir.corpus_bytes:,} bytes ({ir.index_bytes / ir.corpus_bytes:.2%}).")
print(f"Vocabulary size: {ir.vocabulary_size:,} distinct terms.")
print(f"Tokens after preprocessing: {ir.tokens_after:,}.")
print(f"Tokens before preprocessing: {ir.tokens_before:,}.")
for q in QUERIES:
    print(docx_line(ir, q))

Index build time: 0.235055 s
System answer time per query:
network: 0.0234 ms
structure: 0.0427 ms
object: 0.0092 ms
Stop words: CACM collection; 7 words: the, of, a, and, to, is, in.
Preprocessing: HTML tags, citation block, record line and date line removed; lowercase; [a-z]+ tokens; stop words removed.
Index size vs corpus size: 957,888 / 2,217,926 bytes (43.19%).
Vocabulary size: 11,078 distinct terms.
Tokens after preprocessing: 159,105.
Tokens before preprocessing: 374,478.
network — 67 matching documents; top 3: CACM-0757, CACM-1685, CACM-0517; P@3 = 1.00; P@7 = 1.00.
structure — 187 matching documents; top 3: CACM-0115, CACM-1569, CACM-0463; P@3 = 1.00; P@7 = 1.00.
object — 52 matching documents; top 3: CACM-1733, CACM-0598, CACM-1947; P@3 = 1.00; P@7 = 0.57.


## 8. Live demo

In [16]:
ir.search("parallel")[:10]

[('CACM-0141', 0.47443614864652434),
 ('CACM-0392', 0.4217210210191327),
 ('CACM-1302', 0.3795489189172195),
 ('CACM-2685', 0.2777187211589411),
 ('CACM-2182', 0.27110637065515675),
 ('CACM-1367', 0.25303261261147963),
 ('CACM-1158', 0.23721807432326217),
 ('CACM-1795', 0.23721807432326217),
 ('CACM-1262', 0.21383037685477155),
 ('CACM-2896', 0.21086051050956636)]